# 自然语言处理

> 参考：动手学深度学习 v2，第 14 章

本章介绍 NLP 预训练技术的演进脉络：从词级别的静态嵌入（Word2Vec、GloVe）到子词嵌入，再到基于 Transformer 的上下文动态表示（BERT）。这些方法共同构成了现代 NLP 的基石。


---
### 1. 词嵌入（Word2Vec）


**独热编码（One-Hot Encoding）**是最朴素的词表示方式：词表中有 $V$ 个词，每个词用一个 $V$ 维向量表示，该词对应位置为 1，其余为 0。

但独热编码存在致命缺陷：

| 缺陷 | 说明 |
|------|------|
| 维度灾难 | 词表通常有数万到数十万词，向量极度高维稀疏 |
| 无法表达相似性 | 任意两个不同词的余弦相似度均为 0，无法区分语义远近 |
| 无法泛化 | 每个词完全独立，模型无法利用词间的语义关联 |

**词嵌入（Word Embedding）** 将每个词映射到一个低维稠密向量（如 100-300 维），使得语义相似的词在向量空间中距离接近。


> 「分布假说：一个词的含义由其周围词汇的上下文决定。」

这是 Word2Vec 两个模型的理论基石：通过大规模语料中的共现关系学习词的含义。

**1.1 Skip-gram 模型**

**核心思路**：给定**中心词**，预测其周围的**上下文词**。

设中心词为 $w_c$，上下文词为 $w_o$，两者对应的嵌入向量分别为 $\mathbf{v}_c$（中心词向量）和 $\mathbf{u}_o$（上下文词向量）。条件概率通过 softmax 建模：

$$P(w_o \mid w_c) = \frac{\exp(\mathbf{u}_o^\top \mathbf{v}_c)}{\sum_{i \in \mathcal{V}} \exp(\mathbf{u}_i^\top \mathbf{v}_c)}$$

**训练目标**：对语料库中所有中心词-上下文词对，最大化对数似然：

$$\max \sum_{t=1}^{T} \sum_{-m \leq j \leq m,\, j \neq 0} \log P(w^{(t+j)} \mid w^{(t)})$$

其中 $m$ 为上下文窗口半径。

**1.2 CBOW 模型**

**核心思路**：给定周围的**上下文词**，预测**中心词**（与 Skip-gram 方向相反）。

CBOW 将上下文词的向量取平均，再与中心词向量做点积：

$$P(w_c \mid w_{o_1}, \ldots, w_{o_{2m}}) = \frac{\exp\!\left(\mathbf{v}_c^\top \bar{\mathbf{u}}\right)}{\sum_{i \in \mathcal{V}} \exp\!\left(\mathbf{v}_i^\top \bar{\mathbf{u}}\right)}, \quad \bar{\mathbf{u}} = \frac{1}{2m} \sum_{j} \mathbf{u}_{o_j}$$

**1.3 两种模型的对比**

| 对比项 | Skip-gram | CBOW |
|--------|-----------|------|
| 预测方向 | 中心词 → 上下文词 | 上下文词 → 中心词 |
| 训练速度 | 较慢（多个预测目标） | 较快（一个预测目标） |
| 稀有词效果 | 更好（每个样本独立训练） | 稍差 |
| 常用程度 | 更广泛使用 | 较少使用 |

**词向量的几何意义**：训练完成后，词嵌入空间具有良好的代数性质。著名的类比关系：

$$\text{vec}(\text{king}) - \text{vec}(\text{man}) + \text{vec}(\text{woman}) \approx \text{vec}(\text{queen})$$

这说明词向量不仅编码了词义，还隐式捕捉了词间的语义关系。

In [ ]:
import torch
import torch.nn as nn

class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.center_embed = nn.Embedding(vocab_size, embed_dim)   # 中心词向量
        self.context_embed = nn.Embedding(vocab_size, embed_dim)  # 上下文词向量

    def forward(self, center, context_neg):
        # center: (B,)  context_neg: (B, num_context+num_neg)
        v = self.center_embed(center).unsqueeze(2)        # (B, d, 1)
        u = self.context_embed(context_neg)               # (B, N, d)
        return torch.bmm(u, v).squeeze(2)                 # (B, N)

---
### 2. 全局词向量嵌入（GloVe）


Word2Vec 以滑动窗口方式训练，每次只看局部上下文，**没有直接利用语料库的全局统计信息**（如某词对在全局出现的频率）。


GloVe 首先在整个语料库上统计**全局共现计数**：

$$x_{ij} = \text{词 } w_j \text{ 在词 } w_i \text{ 的上下文中出现的总次数}$$

$x_{ij}$ 构成一个 $|\mathcal{V}| \times |\mathcal{V}|$ 的共现矩阵，包含了语料库的完整全局统计信息。

**目标函数**：GloVe 用**加权平方损失**拟合对数共现频率：

$$\min \sum_{i,j \in \mathcal{V}} h(x_{ij}) \left(\mathbf{u}_j^\top \mathbf{v}_i + b_i + c_j - \log x_{ij}\right)^2$$

各项含义：

| 项 | 含义 |
|----|------|
| $\mathbf{u}_j^\top \mathbf{v}_i$ | 上下文词向量与中心词向量的点积 |
| $b_i, c_j$ | 中心词和上下文词的偏置项 |
| $\log x_{ij}$ | 全局共现频率的对数（拟合目标） |
| $h(x_{ij})$ | 权重函数，对罕见共现赋予更小权重 |

权重函数通常取：$h(x) = \min(1, (x/x_{\max})^{\alpha})$，$\alpha=3/4$



由于 $x_{ij} = x_{ji}$（共现是对称的），GloVe 的中心词向量 $\mathbf{v}_i$ 和上下文词向量 $\mathbf{u}_i$ 在数学上是等价的。实践中将两者相加作为词的最终表示：$\mathbf{v}_i + \mathbf{u}_i$。



| 对比项 | Word2Vec | GloVe |
|--------|----------|-------|
| 统计粒度 | 局部窗口 | 全局共现矩阵 |
| 损失函数 | 交叉熵（softmax） | 加权平方损失 |
| 训练方式 | 在线随机梯度 | 批量优化 |
| 词向量对称性 | 中心词/上下文词向量不同 | 两者等价，可相加 |

---
### 3. 子词嵌入


Word2Vec 和 GloVe 都是词级别的模型，面临两个主要问题：

1. **OOV（Out-Of-Vocabulary）**：测试时遇到训练时未见过的词，无法处理
2. **形态信息忽视**：「run」「running」「runner」被视为完全独立的词，   无法利用词形变化中蕴含的语义关联

**子词嵌入**通过在词内部的字符/子词层面建模，解决上述问题。

**3.1 fastText：字符 n-gram 嵌入**

**核心思路**：将一个词表示为其所有字符 n-gram 的向量之和。

例如，对于词「where」（$n=3$），提取：

```
<wh, whe, her, ere, re>, <where>
```

（用尖括号标记词的开始和结束，避免与其他词的 n-gram 混淆）

词向量 = 所有子词向量之和：

$$\mathbf{v}_{\text{where}} = \sum_{g \in \mathcal{G}_{\text{where}}} \mathbf{z}_g$$

**优势**：
- OOV 词可由已见子词组合表示
- 形态相关词（run, running, runner）共享子词向量，天然捕捉形态语义
- 对拼写错误更鲁棒

**3.2 字节对编码（BPE）**

**BPE（Byte Pair Encoding）** 是一种数据驱动的子词分割算法，现已成为 GPT、BERT 等大模型的标准分词方式。

**算法流程**：

```
输入：语料库
初始化：词表 = 所有单个字符 + 特殊符号

重复直到词表达到预设大小：
  1. 统计语料库中所有相邻符号对的出现频率
  2. 合并出现频率最高的符号对为新符号
  3. 将新符号加入词表，更新语料库
```

**举例**：
```
初始：l o w, l o w e r, n e w e s t, w i d e s t
第1次合并（es 频率最高）：l o w, l o w e r, n e w est, w i d est
第2次合并（est 频率最高）：l o w, l o w e r, n e w est, w i d est
...
```

**BPE vs fastText**：

| 对比项 | fastText | BPE |
|--------|----------|-----|
| 子词长度 | 固定 n-gram（3-6字符） | 可变长度（数据驱动） |
| 词表大小 | 较大 | 可精确控制 |
| 现代使用 | 较少 | 主流（GPT/BERT 等）|


In [ ]:
def bpe_tokenize(word, vocab):
    """用学好的 BPE 词表对词进行子词分割"""
    tokens = list(word) + ['</w>']   # 末尾标记词边界
    while len(tokens) > 1:
        # 找到词表中出现的最长子词对
        pairs = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
        merge = next((p for p in vocab if p in pairs), None)
        if merge is None:
            break
        # 合并该对
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == merge:
                new_tokens.append(tokens[i] + tokens[i+1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

---
### 4. 词的相似性与类比任务

**4.1 余弦相似度**

衡量两个词向量 $\mathbf{u}$、$\mathbf{v}$ 的语义相似度，通常用余弦相似度：

$$\text{sim}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u}^\top \mathbf{v}}{\|\mathbf{u}\| \cdot \|\mathbf{v}\|}$$

取值范围 $[-1, 1]$，值越大表示语义越相近。

实际计算时在分母加极小值（如 $10^{-9}$）保证数值稳定性。

**4.2 词类比任务**

词类比任务检验词向量空间的代数结构：如果 $a$ 之于 $b$ 就如 $c$ 之于 $d$，那么：

$$\text{vec}(b) - \text{vec}(a) + \text{vec}(c) \approx \text{vec}(d)$$

在所有词向量中找与 $\text{vec}(b) - \text{vec}(a) + \text{vec}(c)$ 余弦相似度最高的词即为答案 $d$。

经典示例：

| 类比关系 | 向量运算 | 答案 |
|----------|----------|------|
| man:king :: woman:? | vec(king)-vec(man)+vec(woman) | queen |
| China:Beijing :: Japan:? | vec(Beijing)-vec(China)+vec(Japan) | Tokyo |
| good:better :: bad:? | vec(better)-vec(good)+vec(bad) | worse |

**4.3 词向量的几何性质**

预训练词向量在空间中自然形成语义聚类：

- **语义关系**：国家名聚集在一起，职业词聚集在一起
- **句法关系**：动词原形、过去式、进行时构成平行的向量偏移
- **方向性**：性别、数量、时态等属性对应特定的向量方向

这些性质说明词向量不是简单的查找表，而是捕获了语言深层结构的分布式表示。

In [ ]:
import torch
import torch.nn.functional as F

def cosine_sim(u, v):
    return F.cosine_similarity(u.unsqueeze(0), v.unsqueeze(0)).item()

def analogy(a, b, c, embeddings, vocab):
    """解答类比问题 a:b :: c:?"""
    query = embeddings[vocab[b]] - embeddings[vocab[a]] + embeddings[vocab[c]]
    sims = F.cosine_similarity(embeddings, query.unsqueeze(0))
    # 排除 a, b, c 本身
    for w in [a, b, c]:
        sims[vocab[w]] = -1
    return vocab.lookup_token(sims.argmax().item())

---
### 5. BERT：来自Transformer的双向编码器表示


在 BERT 之前，NLP 预训练有两条技术路线，各有缺陷：

| 模型 | 方向 | 缺陷 |
|------|------|------|
| ELMo | 双向（两个单向拼接） | 依赖特定任务架构，迁移不灵活 |
| GPT | 单向（从左到右） | 无法利用右侧上下文，一词多义表示受限 |

**典型问题**：「bank」既可表示「银行」也可表示「河岸」，单向模型在不同语境中对该词生成相同的表示，无法区分。

**5.1 BERT 的核心创新**

BERT（Bidirectional Encoder Representations from Transformers）结合了两者优点：

- **双向编码**（来自 ELMo 的思路）：每个词的表示同时依赖左右两侧上下文
- **任务无关**（来自 GPT 的思路）：统一的预训练 + 微调范式，适配多种下游任务

**5.2 网络架构**

BERT 的主体是多层 **Transformer 编码器**（仅使用编码器，没有解码器）。

| 版本 | 层数 | 隐藏层维度 | 注意力头数 | 参数量 |
|------|------|-----------|-----------|--------|
| BERT-Base | 12 | 768 | 12 | 1.1 亿 |
| BERT-Large | 24 | 1024 | 16 | 3.4 亿 |

**5.3 输入表示：三层嵌入的叠加**

BERT 的输入是三种嵌入的逐元素相加：

$$\text{Input} = \text{TokenEmbed} + \text{SegmentEmbed} + \text{PositionEmbed}$$

**词元嵌入（Token Embedding）**
- 将每个词（子词）映射到稠密向量
- 词表基于 BPE，大小约 30000

**片段嵌入（Segment Embedding）**
- 区分句子对中的两个句子
- 第一句的所有词用 $E_A$，第二句的所有词用 $E_B$

**位置嵌入（Position Embedding）**
- 编码每个位置的信息（最长 512 个词元）
- 与原始 Transformer 不同，BERT 使用**可学习**的位置嵌入（非固定正弦函数）

**输入格式**：
```
单句：  [CLS] 词1 词2 ... 词n [SEP]
句子对：[CLS] 句A ... [SEP] 句B ... [SEP]
```

特殊标记 `[CLS]` 的最终隐藏状态用于句子级任务（如分类），`[SEP]` 分隔两个句子。

**5.4 预训练任务一：掩蔽语言模型（MLM）**

**问题**：双向 Transformer 如果直接预测每个词，等于给模型提供了答案（每个词可以直接看到自己）。

**解决方案**：随机遮蔽 15% 的词元，让模型预测被遮蔽的词。

被选中的 15% 词元按如下策略处理：

| 策略 | 比例 | 原因 |
|------|------|------|
| 替换为 `[MASK]` | 80% | 主要任务 |
| 替换为随机词 | 10% | 迫使模型不过度依赖 `[MASK]` 标记 |
| 保持原词不变 | 10% | 减少预训练与微调的分布差异 |

这种 **80/10/10 策略**避免了「预训练看到 `[MASK]`，微调却从未出现」的不匹配问题。

MLM 使 BERT 能够在双向上下文中理解词义，有效解决一词多义问题。

**5.5 预训练任务二：下一句预测（NSP）**

**动机**：MLM 在词级别训练，但许多任务（如自然语言推理、问答）需要理解句子间的关系。

**任务设计**：
- 50% 的概率：输入一对连续的真实句子（标签：IsNext）
- 50% 的概率：第二句随机替换为其他句子（标签：NotNext）
- 使用 `[CLS]` 的隐藏表示做二分类预测

两个预训练任务都从纯文本语料库中**自动生成**监督信号，无需人工标注。

**5.6 微调范式**

BERT 的下游任务适配遵循统一的「预训练 + 微调」范式：

```
预训练 BERT（大规模无标注语料）
           |
           v
加载预训练权重，在输出层添加任务头
           |
           v
在有标注的目标任务数据上微调全部参数
```

不同下游任务的适配方式：

| 任务类型 | 使用的表示 | 任务头 |
|----------|-----------|--------|
| 句子分类（情感分析） | `[CLS]` 隐藏状态 | 线性分类层 |
| 句对分类（自然语言推理） | `[CLS]` 隐藏状态 | 线性分类层 |
| 序列标注（命名实体识别） | 每个词元的隐藏状态 | 逐词分类层 |
| 抽取式问答 | 每个词元的隐藏状态 | 预测答案的起止位置 |

**关键点**：微调时更新**所有** BERT 参数（而不是冻结），任务头通常很轻量。

In [ ]:
import torch
import torch.nn as nn

class BERTInputEmbedding(nn.Module):
    """BERT 输入层：词元 + 片段 + 位置嵌入之和"""
    def __init__(self, vocab_size, embed_dim, max_len=512, num_segments=2):
        super().__init__()
        self.token_embed   = nn.Embedding(vocab_size, embed_dim)
        self.segment_embed = nn.Embedding(num_segments, embed_dim)
        self.pos_embed     = nn.Embedding(max_len, embed_dim)  # 可学习位置嵌入
        self.norm = nn.LayerNorm(embed_dim)
        self.drop = nn.Dropout(0.1)

    def forward(self, tokens, segments):
        # tokens, segments: (B, seq_len)
        B, L = tokens.shape
        pos = torch.arange(L, device=tokens.device).unsqueeze(0).expand(B, -1)
        x = self.token_embed(tokens) + self.segment_embed(segments) + self.pos_embed(pos)
        return self.drop(self.norm(x))


class BERTForClassification(nn.Module):
    """在 BERT 基础上做句子分类（使用 [CLS] 表示）"""
    def __init__(self, bert, hidden_dim, num_classes):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, tokens, segments):
        hidden = self.bert(tokens, segments)  # (B, seq_len, d)
        cls = hidden[:, 0, :]                 # 取 [CLS] 位置的表示
        return self.classifier(cls)